# Spelling Correction with the Noisy Channel Model

## Task 1: Data Loading and Preprocessing


In [8]:
!pip install pyspellchecker

In [2]:
from google.colab import files
import re
from collections import Counter

# TODO: Upload holbrook-tagged-1.dat
# Hint: files.upload() returns a dict of {filename: bytes}; the key is your file's name
uploaded = files.upload()
data_path = list(uploaded.keys())[0]

# TODO: Read the file's text
# Hint: the dataset has special (non-UTF-8) characters, so open it with encoding='latin-1'
with open(data_path, 'r', encoding='latin-1') as f:
  text = f.read()

# TODO: Write a regex pattern to extract (correct_word, wrong_word) pairs
# Hint: errors in the dataset look like <ERR targ="correct">wrong</ERR>
#       use re.findall with a capturing group for the targ="..." value and one for the tag's inner text
# Note: in this dataset the targ value is sometimes unquoted (targ=sister) rather than
# quoted (targ="sister"), so the pattern captures everything up to the closing '>'
# and we strip stray quotes afterwards.
pattern = r'<ERR targ=([^>]+)>\s*(.*?)\s*</ERR>'
matches = re.findall(pattern, text)

# TODO: Build a vocabulary Counter from every correct/wrong word found
# Hint: lowercase and .strip() each word before counting, and count both
#       the correct spelling and the misspelling so both are in vocab
vocab = Counter()
for correct, wrong in matches:
    correct_clean = correct.strip().strip('"').lower()
    wrong_clean = wrong.strip().lower()
    vocab[correct_clean] += 1
    vocab[wrong_clean] += 1

print(f"Total error annotations found: {len(matches)}")
print(f"Vocabulary size: {len(vocab)}")
print("Example P(c):", list(vocab.items())[:10])

Saving holbrook-tagged-1.dat to holbrook-tagged-1 (1).dat
Total error annotations found: 2600
Vocabulary size: 2613
Example P(c): [('sister', 4), ('siter', 2), ('goes', 2), ('go', 23), ('sometimes', 8), ('some times', 4), ('club', 2), ('clob', 1), ('bellringing', 1), ('bell ringing', 1)]


## Task 2: Building the Noisy Channel Model



In [3]:
def P_c(c):
    # TODO: language model probability P(c)
    # Hint: frequency of c in vocab, divided by total count of all words in vocab
    total = sum(vocab.values())
    return vocab.get(c, 0) / total

def P_w_given_c(w, c):
    # TODO: error model P(w|c) based on inverse edit distance
    # Hint: call edit_distance(w, c) (defined in Task 3) and turn it into a
    #       probability-like score where smaller distance -> higher value,
    #       e.g. 1 / (distance + 1)
    distance = edit_distance(w, c)
    return 1 / (distance + 1)


def noisy_channel_score(w, c):
    # TODO: combine P(w|c) and P(c) as in the formula above
    return P_w_given_c(w, c) * P_c(c)

## Task 3: Edit Distance and Candidate Generation


In [4]:
# Edit Distance function (Levenshtein distance)
def edit_distance(s1, s2):
    m, n = len(s1), len(s2)
    dp = [[0] * (n + 1) for _ in range(m + 1)]

    for i in range(m + 1):
        dp[i][0] = i
    for j in range(n + 1):
        dp[0][j] = j

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            if s1[i - 1] == s2[j - 1]:
                dp[i][j] = dp[i - 1][j - 1]
            else:
                dp[i][j] = 1 + min(
                    dp[i - 1][j],    # deletion
                    dp[i][j - 1],    # insertion
                    dp[i - 1][j - 1] # substitution
                )
    return dp[m][n]

def calculate_edit_distance(s1, s2):

    m, n = len(s1), len(s2)

    # Initialize the DP matrix with zeros. Dimensions are (m+1) x (n+1).
    dp = np.zeros((m + 1, n + 1), dtype=int)

    # --- Step 2: Initialization ---
    # Cost of deletions (transforming s1 to empty string)
    for i in range(m + 1):
        dp[i, 0] = i

    # Cost of insertions (transforming empty string to s2)
    for j in range(n + 1):
        dp[0, j] = j

    # --- Step 3: Fill the Matrix ---
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            # If characters are the same, cost is 0, take diagonal value.
            # Note: string indices are i-1 and j-1 because DP table is 1-indexed.
            if s1[i - 1] == s2[j - 1]:
                cost = 0
            else:
                cost = 1 # Cost for substitution

            # Calculate costs for each operation
            deletion_cost = dp[i - 1, j] + 1
            insertion_cost = dp[i, j - 1] + 1
            substitution_cost = dp[i - 1, j - 1] + cost

            # The value of dp[i,j] is the minimum of these three costs
            dp[i, j] = min(deletion_cost, insertion_cost, substitution_cost)

    # --- Step 4: Final Result ---
    edit_distance = dp[m, n]

    return edit_distance, dp


def generate_candidates(w, max_dist=2):
    # TODO: return the set of vocab words within max_dist edit distance of w
    # Hint: loop over every word in vocab, compute edit_distance(w, word),
    #       keep the word if the distance is <= max_dist
    candidates = set()
    for word in vocab:
        if edit_distance(w, word) <= max_dist:
            candidates.add(word)
    return candidates


## Task 4: Create the Spelling Corrector



In [5]:
def correct_word(w):
    w = w.lower()
    # TODO: if w is already a known/correct word, just return it
    # TODO: otherwise, generate_candidates(w); if there are none, fall back to returning w
    # Hint: pick the candidate that maximizes noisy_channel_score(w, candidate)
    #       (see Python's max() with a key= argument)
    if w in vocab:
        return w

    candidates = generate_candidates(w)
    if not candidates:
        return w

    return max(candidates, key=lambda c: noisy_channel_score(w, c))


def correct_sentence(sentence):
    # TODO: apply correct_word to every word in the sentence and join back with spaces
    # Hint: strip punctuation from each token before correcting it, so "today."
    #       is looked up as "today" rather than as its own vocab entry
    words = sentence.split()
    corrected_words = []
    for word in words:
        stripped = re.sub(r"[^\w']", "", word)
        if not stripped:
            corrected_words.append(word)
            continue
        corrected_words.append(correct_word(stripped))
    return ' '.join(corrected_words)


# Test on given sentences
test_sentences = [
    "I have a good freind who is my best companion.",
    "The wether is pleasent today.",
    "My computer is runing sloowly.",
    "Wile I am not sure about this but still I'll eate this."
]

for sent in test_sentences:
    print("Original:", sent)
    print("Corrected:", correct_sentence(sent))
    print()

Original: I have a good freind who is my best companion.
Corrected: i have a too frend who is my best companion

Original: The wether is pleasent today.
Corrected: the ther is please tod

Original: My computer is runing sloowly.
Corrected: my computer is running slowly

Original: Wile I am not sure about this but still I'll eate this.
Corrected: were i to not sure about their but still i'll eate their



## Task 5: Comparison with a Standard Library

Finally, let's compare your model's performance with a well-established spell-checking library, `pyspellchecker`. This will give you a baseline to understand how your model performs and where it could be improved.

In [6]:
import numpy as np
!pip install pyspellchecker
from spellchecker import SpellChecker

spell = SpellChecker()

test_sentences = [
    "I have a good freind who is my best companion.",
    "The wether is pleasent today.",
    "My computer is runing sloowly.",
    "Wile I am not sure about this but still I'll eate this."
]

for sent in test_sentences:
    words = sent.split()
    print("\nSentence:", sent)
    # TODO: print your model's correction for each word
    # Hint: use correct_word(w) for words not already in vocab, else keep w as-is
    our_correction = correct_sentence(sent)
    print("Our Model Correction:", our_correction)

    # TODO: print pyspellchecker's correction for each word
    # Hint: spell.correction(w) returns the library's suggested fix
    pyspell_words = []
    for word in words:
        stripped = re.sub(r"[^\w']", "", word).lower()
        if not stripped:
            pyspell_words.append(word)
            continue
        suggestion = spell.correction(stripped)
        pyspell_words.append(suggestion if suggestion else stripped)
    print("Pyspellchecker Correction:", ' '.join(pyspell_words))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 47.7 MB/s eta 0:00:00

Sentence: I have a good freind who is my best companion.
Our Model Correction: i have a too frend who is my best companion
Pyspellchecker Correction: i have a good friend who is my best companion

Sentence: The wether is pleasent today.
Our Model Correction: the ther is please tod
Pyspellchecker Correction: the wether is pleasant today

Sentence: My computer is runing sloowly.
Our Model Correction: my computer is running slowly
Pyspellchecker Correction: my computer is running slowly

Sentence: Wile I am not sure about this but still I'll eate this.
Our Model Correction: were i to not sure about their but still i'll eate their
Pyspellchecker Correction: wile i am not sure about this but still i'll hate this
